# Hello Spark

Verbinden met het lokale Spark-cluster via **Spark Connect** (poort 15002), het protocol dat ook Databricks Connect gebruikt.

Vereist: `tilt up` draait en `pf-connect` is groen in de Tilt-UI.

**Catalog-isolatie:** Spark Connect en Spark Thrift draaien als gescheiden Spark-applicaties met elk een eigen in-memory catalog. Tabellen die `dbt-smoke` aanmaakt via Thrift zie je hier dus niet — voor gedeelde state heb je een Hive Metastore nodig (zie v2 in `../README.md`).

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .remote("sc://localhost:15002")
    .getOrCreate()
)
print("Spark version:", spark.version)

In [ ]:
# Eigen DataFrame opbouwen — dit landt op het cluster, niet op je laptop
df = spark.createDataFrame(
    [
        (1, "alice", 100.00, "2026-01-01"),
        (2, "bob",   250.00, "2026-01-02"),
        (3, "alice",  75.50, "2026-01-03"),
        (4, "carol", 500.00, "2026-01-03"),
        (5, "bob",   150.00, "2026-01-04"),
    ],
    ["order_id", "customer_id", "amount", "order_date"],
)
df.createOrReplaceTempView("orders")
df.show()

In [ ]:
# Spark SQL — analyse op het cluster, resultaat naar je notebook
spark.sql("""
    SELECT customer_id,
           COUNT(*)    AS n_orders,
           SUM(amount) AS lifetime_value
    FROM orders
    GROUP BY customer_id
    ORDER BY lifetime_value DESC
""").show()

In [ ]:
# DataFrame-API i.p.v. SQL — zelfde resultaat
from pyspark.sql import functions as F
(
    spark.table("orders")
    .groupBy("customer_id")
    .agg(F.count("*").alias("n_orders"), F.sum("amount").alias("lifetime_value"))
    .orderBy(F.col("lifetime_value").desc())
    .show()
)

In [ ]:
# Resultaat als pandas DataFrame voor verdere analyse / plotten
pdf = (
    spark.table("orders")
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
    .orderBy("order_date")
    .toPandas()
)
pdf